This notebook contains a number of uncertainty quantification analyses, which use a trained Bayesian Factorization Machine model 

In [13]:
import numpy as np, pandas as pd, joblib
import os
from pathlib import Path
from dataloaders.load_ecotox import load_ecotox_data   # ← same loader

ART = Path("artifacts")      # directory with the .npy files

# 1 ── reload the data in exactly the same order used for training
DATA_DIR = Path("/home/tad/Desktop/Thesisfiles/ThesisCode/ecotox-toolkit/data_files")
full_data, y_centered = load_ecotox_data(
    adore_path      = DATA_DIR / "ecotox_mortality_processed.csv",
    chemicals_path  = DATA_DIR / "ecotox_properties_with-oecd-function.csv",
    shuffle         = True,
    random_state    = 42,
    use_selfies     = False,
    use_mol2vec     = False,
    use_fingerprint = False,
)

# 2 ── attach predictions
oof_mean = np.load(ART / "oof_mean.npy")
oof_var  = np.load(ART / "oof_var.npy")
assert len(full_data) == oof_mean.size == oof_var.size        # sanity

full_data = full_data.copy()
full_data["pred_mean"] = oof_mean.astype(np.float32)
full_data["pred_sd"]   = np.sqrt(oof_var, dtype=np.float32)
full_data["ci_lo"]     = full_data["pred_mean"] - 1.96 * full_data["pred_sd"]
full_data["ci_hi"]     = full_data["pred_mean"] + 1.96 * full_data["pred_sd"]

# 3 ── grouped summary (work-around: keep as_index=True, reset_index later)
summary = (
    full_data
    .groupby(["species", "duration", "CAS"])       # leave as_index=True
    .agg(
        n_obs       = ("pred_mean", "size"),
        mean_pred   = ("pred_mean", "mean"),
        median_pred = ("pred_mean", "median"),
        mean_sd     = ("pred_sd",   "mean"),
    )
    .reset_index()                                 # put keys back as columns
    .sort_values("mean_pred")
)

print(summary.head(10))


/tmp/ipykernel_15458/541646773.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["species", "duration", "CAS"])       # leave as_index=True


                            species  duration         CAS  n_obs  mean_pred  \
12018786         Palaemonetes_pugio      96.0  52918-63-5      2  -6.011371   
6997987              Gammarus_pulex      96.0  76703-62-3      2  -5.988581   
7590306          Homarus_americanus      96.0  52918-63-5      6  -5.865626   
7650397             Hyalella_azteca      48.0  76703-62-3     12  -5.826257   
11979227    Palaemonetes_argentinus      96.0  52315-07-8      1  -5.767113   
657626           Americamysis_bahia      96.0  52918-63-5      3  -5.744193   
7650784             Hyalella_azteca      48.0  91465-08-6     12  -5.625848   
6984807     Gammarus_pseudolimnaeus      96.0  76703-62-3      3  -5.615839   
647741           Americamysis_bahia      24.0  52918-63-5      1  -5.599671   
9053267   Macrobrachium_rosenbergii      96.0  52315-07-8      1  -5.538266   

          median_pred   mean_sd  
12018786    -6.011371  0.174271  
6997987     -5.988581  0.157639  
7590306     -5.899955  0.218

In [11]:
DATA_DIR = "/home/tad/Desktop/Thesisfiles/ThesisCode/ecotox-toolkit/data_files"
adore_path = os.path.join(DATA_DIR, "ecotox_mortality_processed.csv")
chemicals_path = os.path.join(DATA_DIR, "ecotox_properties_with-oecd-function.csv")

full_data, y_centered = load_ecotox_data(
adore_path=adore_path,
chemicals_path=chemicals_path,
use_selfies=False,
use_mol2vec=False,
use_fingerprint=False,
shuffle=True,
random_state=42,
)


In [12]:
full_data

,test_id,reference_number,CAS,test_location,test_exposure_type,test_control_type,test_media_type,test_application_freq_unit,test_organism_lifestage,result_id,...,chem_mordred_WPol,chem_mordred_Zagreb1,chem_mordred_Zagreb2,chem_mordred_mZagreb2,DTXSID,oecd_function_prevalent,oecd_function_all,funcuse_prevalent,funcuse_all,conc_centered
0,1076392,5940,552-89-6,LAB,F,I,FW,CON,NR,115475,...,14,50.0,56.0,2.638889,NaN,NaN,NaN,NaN,NaN,1.294115
1,1147015,12427,2702-72-9,LAB,S,C,FW,X,NR,101573,...,15,60.0,65.0,2.944444,NaN,NaN,NaN,NaN,NaN,1.643887
2,2291766,863,123-86-4,LAB,S,C,SW,X,NR,2680607,...,5,28.0,26.0,2.083333,DTXSID3021982,solvent - surfactant,fragrance - solvent - surfactant,solubilizer - solvent,cosmetic - flavoring - fragrance - odorant - s...,2.355409
3,1001083,182,94-09-7,LAB,S,I,FW,X,NR,112388,...,14,54.0,59.0,2.861111,DTXSID8021804,UV stabilizer - other,UV stabilizer - other,active - oral pain reliever - analgesic,active oral pain reliever - active - oral pai...,1.413871
4,2236363,7199,13171-21-6,LAB,S,V,FW,X,NR,2507183,...,27,80.0,90.0,4.458333,DTXSID7021156,biocide,biocide,insecticide,insecticide,2.454773
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70665,1035679,2188,1114-71-2,LAB,S,NR,SW,X,JV,185392,...,14,50.0,52.0,3.444444,NaN,NaN,NaN,NaN,NaN,0.640951
70666,2059172,344,95-06-7,LAB,S,K,FW,X,NC,2100640,...,13,48.0,50.0,3.027778,NaN,NaN,NaN,NaN,NaN,-0.135830
70667,1304436,115741,63-25-2,LAB,R,M,SW,DLY,NR,761019,...,21,74.0,85.0,3.472222,DTXSID9020247,biocide,biocide,insecticide,insecticide - pesticide,-1.786213
70668,1178837,15570,563-12-2,LAB,NR,NR,FW,NR,NR,59617,...,24,82.0,88.0,4.750000,DTXSID2024086,biocide,biocide,insecticide,insecticide,-0.390165
